<a href="https://colab.research.google.com/github/ingridevv/data-engineering-zoomcamp/blob/main/week_6_batch/3_groupby_joins.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Preparing Yellow and Green Taxi Data

In [1]:
%%writefile download_data.sh
set -e

TAXI_TYPE=$1 # "yellow"
YEAR=$2 # 2020

URL_PREFIX="https://github.com/DataTalksClub/nyc-tlc-data/releases/download"

for MONTH in {1..12}; do
  FMONTH=`printf "%02d" ${MONTH}`

  URL="${URL_PREFIX}/${TAXI_TYPE}/${TAXI_TYPE}_tripdata_${YEAR}-${FMONTH}.csv.gz"

  LOCAL_PREFIX="data/raw/${TAXI_TYPE}/${YEAR}/${FMONTH}"
  LOCAL_FILE="${TAXI_TYPE}_tripdata_${YEAR}_${FMONTH}.csv.gz"
  LOCAL_PATH="${LOCAL_PREFIX}/${LOCAL_FILE}"

  echo "downloading ${URL} to ${LOCAL_PATH}"
  mkdir -p ${LOCAL_PREFIX}
  wget ${URL} -O ${LOCAL_PATH}

done

Writing download_data.sh


In [2]:
!chmod +x download_data.sh

In [3]:
# Download yellow taxi data 2020
!./download_data.sh yellow 2020

downloading https://github.com/DataTalksClub/nyc-tlc-data/releases/download/yellow/yellow_tripdata_2020-01.csv.gz to data/raw/yellow/2020/01/yellow_tripdata_2020_01.csv.gz
--2026-03-08 14:03:48--  https://github.com/DataTalksClub/nyc-tlc-data/releases/download/yellow/yellow_tripdata_2020-01.csv.gz
Resolving github.com (github.com)... 140.82.116.4
Connecting to github.com (github.com)|140.82.116.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/513814948/1bc3de9c-af03-46e3-baa6-c4475782682d?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-08T15%3A03%3A07Z&rscd=attachment%3B+filename%3Dyellow_tripdata_2020-01.csv.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-08T14%3A03%3A05Z&ske=2026-03-08T15%3A03%3A07Z&sks=b&skv=2018-11-09&sig=L2ECJKFnQK9ObaJjhZ79a%2Brh8kc4xSGa%2Fw39jcxZ9CU%3D&jwt=eyJ0eXAiOiJKV1QiLC

In [4]:
!ls data/raw/yellow/2020/01

yellow_tripdata_2020_01.csv.gz


In [5]:
# Download yellow taxi data 2021
!./download_data.sh yellow 2021

downloading https://github.com/DataTalksClub/nyc-tlc-data/releases/download/yellow/yellow_tripdata_2021-01.csv.gz to data/raw/yellow/2021/01/yellow_tripdata_2021_01.csv.gz
--2026-03-08 14:04:02--  https://github.com/DataTalksClub/nyc-tlc-data/releases/download/yellow/yellow_tripdata_2021-01.csv.gz
Resolving github.com (github.com)... 140.82.116.3
Connecting to github.com (github.com)|140.82.116.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/513814948/f6895842-79e6-4a43-9458-e5b0b454a340?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-08T14%3A51%3A48Z&rscd=attachment%3B+filename%3Dyellow_tripdata_2021-01.csv.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-08T13%3A51%3A29Z&ske=2026-03-08T14%3A51%3A48Z&sks=b&skv=2018-11-09&sig=LYPL%2FwlLsB%2Bz%2FHaUY%2FMN2mQdX%2BZURO8SYyQglO1vyiY%3D&jwt=eyJ0eXAiOiJK

In [6]:
!zcat data/raw/yellow/2020/01/yellow_tripdata_2020_01.csv.gz | head -n 10

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge
1,2020-01-01 00:28:15,2020-01-01 00:33:03,1,1.20,1,N,238,239,1,6,3,0.5,1.47,0,0.3,11.27,2.5
1,2020-01-01 00:35:39,2020-01-01 00:43:04,1,1.20,1,N,239,238,1,7,3,0.5,1.5,0,0.3,12.3,2.5
1,2020-01-01 00:47:41,2020-01-01 00:53:52,1,.60,1,N,238,238,1,6,3,0.5,1,0,0.3,10.8,2.5
1,2020-01-01 00:55:23,2020-01-01 01:00:14,1,.80,1,N,238,151,1,5.5,0.5,0.5,1.36,0,0.3,8.16,0
2,2020-01-01 00:01:58,2020-01-01 00:04:16,1,.00,1,N,193,193,2,3.5,0.5,0.5,0,0,0.3,4.8,0
2,2020-01-01 00:09:44,2020-01-01 00:10:37,1,.03,1,N,7,193,2,2.5,0.5,0.5,0,0,0.3,3.8,0
2,2020-01-01 00:39:25,2020-01-01 00:39:29,1,.00,1,N,193,193,1,2.5,0.5,0.5,0.01,0,0.3,3.81,0
2,2019-12-18 15:27:49,2019-12-18 15:28:59,1,.00,5,N,193,193,1,0.01,0,0,0,0,0.3,2.81,2.5
2,2019-12-18 15:30:35,2019-1

In [7]:
! ./download_data.sh green 2020

downloading https://github.com/DataTalksClub/nyc-tlc-data/releases/download/green/green_tripdata_2020-01.csv.gz to data/raw/green/2020/01/green_tripdata_2020_01.csv.gz
--2026-03-08 14:04:10--  https://github.com/DataTalksClub/nyc-tlc-data/releases/download/green/green_tripdata_2020-01.csv.gz
Resolving github.com (github.com)... 140.82.116.3
Connecting to github.com (github.com)|140.82.116.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/513814948/6e8a7b28-b1ea-455e-9803-cd8943d96c26?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-08T14%3A50%3A38Z&rscd=attachment%3B+filename%3Dgreen_tripdata_2020-01.csv.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-08T13%3A50%3A06Z&ske=2026-03-08T14%3A50%3A38Z&sks=b&skv=2018-11-09&sig=bmLykUtJkW2%2B0lFNuHdOIjd41Xcq44IZoW3iXUOC%2B9c%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciO

In [8]:
!./download_data.sh green 2021

downloading https://github.com/DataTalksClub/nyc-tlc-data/releases/download/green/green_tripdata_2021-01.csv.gz to data/raw/green/2021/01/green_tripdata_2021_01.csv.gz
--2026-03-08 14:04:14--  https://github.com/DataTalksClub/nyc-tlc-data/releases/download/green/green_tripdata_2021-01.csv.gz
Resolving github.com (github.com)... 140.82.116.3
Connecting to github.com (github.com)|140.82.116.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/513814948/ea387a15-484c-469b-860d-3382ee7659be?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-08T15%3A04%3A27Z&rscd=attachment%3B+filename%3Dgreen_tripdata_2021-01.csv.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-08T14%3A03%3A54Z&ske=2026-03-08T15%3A04%3A27Z&sks=b&skv=2018-11-09&sig=C%2B8XEpERtCbxHKKOUaH2HnkviG8bBrXs779evxTsYvs%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJ

In [9]:
!sudo apt-get install tree

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  tree
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 47.9 kB of archives.
After this operation, 116 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tree amd64 2.0.2-1 [47.9 kB]
Fetched 47.9 kB in 0s (163 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package tree.
(Reading database ... 117540 files and directories currently installe

In [10]:
!tree data

data
└── raw
    ├── green
    │   ├── 2020
    │   │   ├── 01
    │   │   │   └── green_tripdata_2020_01.csv.gz
    │   │   ├── 02
    │   │   │   └── green_tripdata_2020_02.csv.gz
    │   │   ├── 03
    │   │   │   └── green_tripdata_2020_03.csv.gz
    │   │   ├── 04
    │   │   │   └── green_tripdata_2020_04.csv.gz
    │   │   ├── 05
    │   │   │   └── green_tripdata_2020_05.csv.gz
    │   │   ├── 06
    │   │   │   └── green_tripdata_2020_06.csv.gz
    │   │   ├── 07
    │   │   │   └── green_tripdata_2020_07.csv.gz
    │   │   ├── 08
    │   │   │   └── green_tripdata_2020_08.csv.gz
    │   │   ├── 09
    │   │   │   └── green_tripdata_2020_09.csv.gz
    │   │   ├── 10
    │   │   │   └── green_tripdata_2020_10.csv.gz
    │   │   ├── 11
    │   │   │   └── green_tripdata_2020_11.csv.gz
    │   │   └── 12
    │   │       └── green_tripdata_2020_12.csv.gz
    │   └── 2021
    │       ├── 01
    │       │   └── green_tripdata_2021_01.csv.gz
    │       ├── 02
    │       │   └── gre

## Defining Schema

In [11]:
import pyspark
from pyspark.sql import SparkSession

In [12]:
# Initialize spark session
spark = SparkSession.builder \
    .appName('spark_session_02') \
    .getOrCreate()

In [13]:
from pyspark.sql import types

In [14]:
green_schema = types.StructType ([
    types.StructField('VendorID', types.IntegerType(), True),
    types.StructField('lpep_pickup_datetime', types.TimestampType(), True),
    types.StructField('lpep_dropoff_datetime', types.TimestampType(), True),
    types.StructField('store_and_fwd_flag', types.StringType(), True),
    types.StructField('RatecodeID', types.IntegerType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('passenger_count', types.IntegerType(), True),
    types.StructField('trip_distance', types.DoubleType(), True),
    types.StructField('fare_amount', types.DoubleType(), True),
    types.StructField('extra', types.DoubleType(), True),
    types.StructField('mta_tax', types.DoubleType(), True),
    types.StructField('tip_amount', types.DoubleType(), True),
    types.StructField('tolls_amount', types.DoubleType(), True),
    types.StructField('ehail_fee', types.DoubleType(), True),
    types.StructField('improvement_surcharge', types.DoubleType(), True),
    types.StructField('total_amount', types.DoubleType(), True),
    types.StructField('payment_type', types.IntegerType(), True),
    types.StructField('trip_type', types.IntegerType(), True),
    types.StructField('congestion_surcharge', types.DoubleType(), True)
])

In [15]:
year = 2020

for month in range(1, 13):
  print(f'Data is being processed: {year}/{month}')
  input_path = f"data/raw/green/{year}/{month:02d}/"
  output_path = f"data/parquet/green/{year}/{month:02d}/"

df_green = spark.read \
    .option("header", "true") \
    .schema(green_schema) \
    .csv(input_path)

df_green \
        .repartition(4)\
        .write.parquet(output_path)

Data is being processed: 2020/1
Data is being processed: 2020/2
Data is being processed: 2020/3
Data is being processed: 2020/4
Data is being processed: 2020/5
Data is being processed: 2020/6
Data is being processed: 2020/7
Data is being processed: 2020/8
Data is being processed: 2020/9
Data is being processed: 2020/10
Data is being processed: 2020/11
Data is being processed: 2020/12


In [16]:
df_green.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- lpep_pickup_datetime: timestamp (nullable = true)
 |-- lpep_dropoff_datetime: timestamp (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- trip_type: integer (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



In [17]:
import pandas as pd

In [18]:
df_green_pd = pd.read_csv('data/raw/green/2021/01/green_tripdata_2021_01.csv.gz', nrows=1000)

In [19]:
df_green_pd

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
0,2,2021-01-01 00:15:56,2021-01-01 00:19:52,N,1,43,151,1,1.01,5.5,0.5,0.5,0.00,0.00,NaN,0.3,6.80,2,1,0.00
1,2,2021-01-01 00:25:59,2021-01-01 00:34:44,N,1,166,239,1,2.53,10.0,0.5,0.5,2.81,0.00,NaN,0.3,16.86,1,1,2.75
2,2,2021-01-01 00:45:57,2021-01-01 00:51:55,N,1,41,42,1,1.12,6.0,0.5,0.5,1.00,0.00,NaN,0.3,8.30,1,1,0.00
3,2,2020-12-31 23:57:51,2021-01-01 00:04:56,N,1,168,75,1,1.99,8.0,0.5,0.5,0.00,0.00,NaN,0.3,9.30,2,1,0.00
4,2,2021-01-01 00:16:36,2021-01-01 00:16:40,N,2,265,265,3,0.00,-52.0,0.0,-0.5,0.00,0.00,NaN,-0.3,-52.80,3,1,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,2,2021-01-02 12:48:43,2021-01-02 13:54:15,N,1,15,177,1,25.76,77.0,0.0,0.5,2.75,12.24,NaN,0.3,92.79,1,1,0.00
996,1,2021-01-02 12:16:47,2021-01-02 12:24:56,N,1,116,74,1,2.00,9.0,0.0,0.5,1.95,0.00,NaN,0.3,11.75,1,1,0.00
997,2,2021-01-02 12:07:17,2021-01-02 12:12:06,N,1,42,41,1,0.87,5.5,0.0,0.5,0.00,0.00,NaN,0.3,6.30,2,1,0.00
998,2,2021-01-02 12:32:45,2021-01-02 12:59:46,N,1,82,218,1,11.88,35.0,0.0,0.5,0.00,0.00,NaN,0.3,35.80,2,1,0.00


In [20]:
# Adjust file types
spark.createDataFrame(df_green_pd).schema

StructType([StructField('VendorID', LongType(), True), StructField('lpep_pickup_datetime', StringType(), True), StructField('lpep_dropoff_datetime', StringType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('RatecodeID', LongType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('ehail_fee', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('payment_type', LongType(), True), StructField('trip_type', LongType(), True), StructField('congestion_surcharge', DoubleType(), True)])

## Yellow Data Schema

In [21]:
df_yellow_pd = pd.read_csv("data/raw/yellow/2021/01/yellow_tripdata_2021_01.csv.gz", nrows=1000)

In [22]:
spark.createDataFrame(df_yellow_pd).schema

StructType([StructField('VendorID', LongType(), True), StructField('tpep_pickup_datetime', StringType(), True), StructField('tpep_dropoff_datetime', StringType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True)])

In [23]:
yellow_schema = types.StructType ([
      types.StructField('VendorID', types.IntegerType(), True),
      types.StructField('tpep_pickup_datetime', types.TimestampType(), True),
      types.StructField('tpep_dropoff_datetime', types.TimestampType(), True),
      types.StructField('passenger_count', types.IntegerType(), True),
      types.StructField('trip_distance', types.DoubleType(), True),
      types.StructField('RatecodeID', types.IntegerType(), True),
      types.StructField('store_and_fwd_flag', types.StringType(), True),
      types.StructField('PULocationID', types.IntegerType(), True),
      types.StructField('DOLocationID', types.IntegerType(), True),
      types.StructField('payment_type', types.IntegerType(), True),
      types.StructField('fare_amount', types.DoubleType(), True),
      types.StructField('extra', types.DoubleType(), True),
      types.StructField('mta_tax', types.DoubleType(), True),
      types.StructField('tip_amount', types.DoubleType(), True),
      types.StructField('tolls_amount', types.DoubleType(), True),
      types.StructField('improvement_surcharge', types.DoubleType(), True),
      types.StructField('total_amount', types.DoubleType(), True),
      types.StructField('congestion_surcharge', types.DoubleType(), True)
      ])

In [24]:
df_yellow = spark.read \
            .option("header", "true") \
            .schema(yellow_schema) \
            .csv("data/raw/yellow/2021/01/")

In [25]:
year = 2020

for month in range(1, 13):
  print(f'Data is being processed: {year}/{month}')
  input_path = f"data/raw/yellow/{year}/{month:02d}/"
  output_path = f"data/parquet/yellow/{year}/{month:02d}/"

df_yellow = spark.read \
    .option("header", "true") \
    .schema(yellow_schema) \
    .csv(input_path)

df_yellow \
        .repartition(4)\
        .write.parquet(output_path)

Data is being processed: 2020/1
Data is being processed: 2020/2
Data is being processed: 2020/3
Data is being processed: 2020/4
Data is being processed: 2020/5
Data is being processed: 2020/6
Data is being processed: 2020/7
Data is being processed: 2020/8
Data is being processed: 2020/9
Data is being processed: 2020/10
Data is being processed: 2020/11
Data is being processed: 2020/12


In [26]:
!tree data

data
├── parquet
│   ├── green
│   │   └── 2020
│   │       └── 12
│   │           ├── part-00000-737b42e2-a328-4b94-9041-372bb1cf0330-c000.snappy.parquet
│   │           ├── part-00001-737b42e2-a328-4b94-9041-372bb1cf0330-c000.snappy.parquet
│   │           ├── part-00002-737b42e2-a328-4b94-9041-372bb1cf0330-c000.snappy.parquet
│   │           ├── part-00003-737b42e2-a328-4b94-9041-372bb1cf0330-c000.snappy.parquet
│   │           └── _SUCCESS
│   └── yellow
│       └── 2020
│           └── 12
│               ├── part-00000-170eeb02-93ea-4652-a0f5-be88a3a6eacc-c000.snappy.parquet
│               ├── part-00001-170eeb02-93ea-4652-a0f5-be88a3a6eacc-c000.snappy.parquet
│               ├── part-00002-170eeb02-93ea-4652-a0f5-be88a3a6eacc-c000.snappy.parquet
│               ├── part-00003-170eeb02-93ea-4652-a0f5-be88a3a6eacc-c000.snappy.parquet
│               └── _SUCCESS
└── raw
    ├── green
    │   ├── 2020
    │   │   ├── 01
    │   │   │   └── green_tripdata_2020_01.csv.gz
    │   │   

# SQL with Spark

In [27]:
# Read green parquet files
df_green_parquet = spark.read.parquet('data/parquet/green/*/*')

In [28]:
# rename dropoff and pickup time

df_green_parquet = df_green_parquet \
              .withColumnRenamed('lpep_pickup_datetime', 'pickup_datetime') \
              .withColumnRenamed('lpep_dropoff_datetime', 'dropoff_datetime')

In [29]:
df_green_parquet.show()

+--------+-------------------+-------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|VendorID|    pickup_datetime|   dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|congestion_surcharge|
+--------+-------------------+-------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|       2|2020-12-22 16:05:01|2020-12-22 18:34:59|                 N|         1|         189|         163|              5|         3.73|       24.0|  1.0|    0.5|      5.16

In [30]:
df_yellow_parquet = spark.read.parquet('data/parquet/yellow/*/*')

In [31]:
df_yellow_parquet = df_yellow_parquet \
                  .withColumnRenamed('tpep_pickup_datetime', 'pickup_datetime') \
                  .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime')

In [32]:
common_columns = []

yellow_columns = set(df_yellow_parquet.columns)

for col in df_green_parquet.columns:
  if col in df_yellow_parquet.columns:
    common_columns.append(col)


In [33]:
df_yellow_parquet

DataFrame[VendorID: int, pickup_datetime: timestamp, dropoff_datetime: timestamp, passenger_count: int, trip_distance: double, RatecodeID: int, store_and_fwd_flag: string, PULocationID: int, DOLocationID: int, payment_type: int, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double]

In [34]:
common_columns

['VendorID',
 'pickup_datetime',
 'dropoff_datetime',
 'store_and_fwd_flag',
 'RatecodeID',
 'PULocationID',
 'DOLocationID',
 'passenger_count',
 'trip_distance',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'payment_type',
 'congestion_surcharge']

In [35]:
from pyspark.sql import functions as F

In [36]:
df_green_service = df_green_parquet\
                .select(common_columns) \
                .withColumn('service_type', F.lit('green'))

In [37]:
df_yellow_service = df_yellow_parquet\
                .select(common_columns) \
                .withColumn('service_type', F.lit('yellow'))

In [38]:
df_trips_data = df_green_service.unionAll(df_yellow_service)

In [39]:
df_trips_data.groupBy('service_type').count().show()

+------------+-------+
|service_type|  count|
+------------+-------+
|       green|  83130|
|      yellow|1461897|
+------------+-------+



## Querying the Data

In [40]:
df_trips_data.registerTempTable("trips_data")

/usr/local/lib/python3.12/dist-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [41]:
spark.sql(
  """
    SELECT
      service_type,
      count(1)
    FROM
      trips_data
    GROUP BY service_type
  """
).show()

+------------+--------+
|service_type|count(1)|
+------------+--------+
|       green|   83130|
|      yellow| 1461897|
+------------+--------+



In [42]:
df_nyc_taxi = spark.sql (
    """
      SELECT
      -- Revenue grouping
      PULocationID AS revenue_zone,
      date_trunc('month', pickup_datetime) AS revenue_month,
      service_type,

      -- Revenue calculation
      SUM(fare_amount) AS revenue_monthly_fare,
      SUM(extra) AS revenue_monthly_extra,
      SUM(mta_tax) AS revenue_monthly_mta_tax,
      SUM(tip_amount) AS revenue_monthly_tip_amount,
      SUM(tolls_amount) AS revenue_monthly_tolls_amount,
      SUM(improvement_surcharge) AS revenue_monthly_improvement_surcharge,
      SUM(total_amount) AS revenue_monthly_total_amount,
      SUM(congestion_surcharge) AS revenue_monthly_congestion_surcharge,

      -- Additional calculations
      AVG(passenger_count) AS avg_monthly_passenger_count,
      AVG(trip_distance) AS avg_monthly_trip_distance
  FROM
      trips_data
  GROUP BY
      1, 2, 3
  """)

In [43]:
df_nyc_taxi.show()

+------------+-------------------+------------+--------------------+---------------------+-----------------------+--------------------------+----------------------------+-------------------------------------+----------------------------+------------------------------------+---------------------------+-------------------------+
|revenue_zone|      revenue_month|service_type|revenue_monthly_fare|revenue_monthly_extra|revenue_monthly_mta_tax|revenue_monthly_tip_amount|revenue_monthly_tolls_amount|revenue_monthly_improvement_surcharge|revenue_monthly_total_amount|revenue_monthly_congestion_surcharge|avg_monthly_passenger_count|avg_monthly_trip_distance|
+------------+-------------------+------------+--------------------+---------------------+-----------------------+--------------------------+----------------------------+-------------------------------------+----------------------------+------------------------------------+---------------------------+-------------------------+
|         131

In [44]:
# Save results in parquet

df_nyc_taxi.coalesce(1).write.parquet("data/report/revenue/", mode="overwrite")

# GroupBy & Joins

In [56]:
!pip install pyngrok

In [57]:

# Acess SparkUI with Ngrok endpoint
from pyngrok import ngrok, conf
import getpass

conf.get_default().auth_token = getpass.getpass()

ui_port = 4040
public_url = ngrok.connect(ui_port).public_url
print(f" * ngrok tunnel \"{public_url}\" -> \"http://127.0.0.1:{ui_port}\"")

··········


 * ngrok tunnel "https://senaida-interlobular-franklyn.ngrok-free.dev" -> "http://127.0.0.1:4040"


In [45]:
df_green.registerTempTable("green")

/usr/local/lib/python3.12/dist-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [54]:
df_green_result = spark.sql (
    """
      SELECT
      PULocationID AS zone,
      date_trunc('hour', lpep_pickup_datetime) AS pickup_hour,
      SUM(total_amount) AS revenue_monthly_total_amount,
      COUNT(1) AS number_records
  FROM
      green
  WHERE lpep_pickup_datetime >= '2020-01-01 00:00:00'
  GROUP BY
      1, 2
  ORDER BY
      1, 2
  """)

In [59]:
df_green_result.show()

+----+-------------------+----------------------------+--------------+
|zone|        pickup_hour|revenue_monthly_total_amount|number_records|
+----+-------------------+----------------------------+--------------+
|   1|2020-12-30 15:00:00|                      120.39|             1|
|   3|2020-12-01 07:00:00|                       30.32|             1|
|   3|2020-12-01 09:00:00|                       45.86|             2|
|   3|2020-12-01 10:00:00|                       16.67|             1|
|   3|2020-12-01 15:00:00|                        12.0|             1|
|   3|2020-12-02 05:00:00|                       62.87|             1|
|   3|2020-12-02 09:00:00|                       88.81|             3|
|   3|2020-12-02 10:00:00|                       78.17|             3|
|   3|2020-12-02 11:00:00|                       22.01|             1|
|   3|2020-12-02 13:00:00|                       22.01|             1|
|   3|2020-12-02 17:00:00|                       25.84|             1|
|   3|

In [60]:
df_green_result.write.parquet('data/report/revenue/green', mode='overwrite')

In [62]:
df_yellow.registerTempTable('yellow')

/usr/local/lib/python3.12/dist-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [63]:
df_yellow_result = spark.sql(
    """
    SELECT
      PULocationID AS zone,
      date_trunc('hour', tpep_pickup_datetime) AS pickup_hour,
      SUM(total_amount) AS revenue_monthly_total_amount,
      COUNT(1) AS number_records
  FROM
      yellow
  WHERE tpep_pickup_datetime >= '2020-01-01 00:00:00'
  GROUP BY
      1, 2
  ORDER BY
      1, 2

    """
)

In [64]:
df_yellow_result.show()

+----+-------------------+----------------------------+--------------+
|zone|        pickup_hour|revenue_monthly_total_amount|number_records|
+----+-------------------+----------------------------+--------------+
|   1|2020-12-02 10:00:00|                       10.81|             1|
|   1|2020-12-03 11:00:00|                       72.92|             3|
|   1|2020-12-03 17:00:00|                       72.36|             1|
|   1|2020-12-04 14:00:00|                        70.3|             1|
|   1|2020-12-06 13:00:00|                        76.3|             1|
|   1|2020-12-07 03:00:00|                      114.36|             1|
|   1|2020-12-08 08:00:00|                       82.55|             1|
|   1|2020-12-08 13:00:00|                       68.46|             1|
|   1|2020-12-10 11:00:00|                      151.56|             1|
|   1|2020-12-10 16:00:00|                         0.3|             1|
|   1|2020-12-11 11:00:00|                        85.3|             1|
|   1|

In [67]:
df_yellow_result \
.repartition(20) \
.write.parquet('data/report/revenue/yellow', mode='overwrite')

In [78]:
df_green_result_tmp = df_green_result \
  .withColumnRenamed('amount', 'green_amount') \
  .withColumnRenamed('number_records', 'green_number_records') \
  .withColumnRenamed('revenue_monthly_total_amount', 'green_monthly_revenue')

df_yellow_result_tmp = df_yellow_result \
          .withColumnRenamed('amount', 'yellow_amount') \
          .withColumnRenamed('number_records', 'yellow_number_records') \
          .withColumnRenamed('revenue_monthly_total_amount', 'yellow_monthly_revenue')

In [79]:
df_join = df_green_result_tmp.join(df_yellow_result_tmp, on=['pickup_hour', 'zone'], how='outer')

In [80]:
df_join.write.parquet('data/report/revenue/total', mode='overwrite')

In [81]:
df_join.show()

+-------------------+----+---------------------+--------------------+----------------------+---------------------+
|        pickup_hour|zone|green_monthly_revenue|green_number_records|yellow_monthly_revenue|yellow_number_records|
+-------------------+----+---------------------+--------------------+----------------------+---------------------+
|2020-12-02 10:00:00|   1|                 NULL|                NULL|                 10.81|                    1|
|2020-12-03 11:00:00|   1|                 NULL|                NULL|                 72.92|                    3|
|2020-12-04 14:00:00|   1|                 NULL|                NULL|                  70.3|                    1|
|2020-12-07 03:00:00|   1|                 NULL|                NULL|                114.36|                    1|
|2020-12-08 08:00:00|   1|                 NULL|                NULL|                 82.55|                    1|
|2020-12-08 13:00:00|   1|                 NULL|                NULL|           

In [88]:
# Download Taxi Zones for Joins
!mkdir -p data/zones

# lookup_taxi_zones
!wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/misc/taxi_zone_lookup.csv -O data/zones/taxi_zone_lookup.csv

--2026-03-08 15:08:36--  https://github.com/DataTalksClub/nyc-tlc-data/releases/download/misc/taxi_zone_lookup.csv
Resolving github.com (github.com)... 140.82.116.3
Connecting to github.com (github.com)|140.82.116.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/513814948/5a2cc2f5-b4cd-4584-9c62-a6ea97ed0e6a?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-08T15%3A46%3A11Z&rscd=attachment%3B+filename%3Dtaxi_zone_lookup.csv&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-08T14%3A46%3A11Z&ske=2026-03-08T15%3A46%3A11Z&sks=b&skv=2018-11-09&sig=3jFHrldrz3WXz2ls9LdVoiZayDBYAFKyF51v%2B1iYahY%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3Mjk4MjgxNiwibmJmIjoxNzcyOTgyNTE2LCJwYXRoIjoicmVsZWFzZWFzc2V0c

In [89]:
df_zones = spark.read \
      .option("header", "true") \
      .csv('data/zones/taxi_zone_lookup.csv')

In [90]:
df_zones_final = df_yellow_result_tmp.join(df_zones, df_yellow_result_tmp.zone == df_zones.LocationID)

In [94]:
df_zones_final.drop('LocationID', 'zone').write.parquet('data/zones', mode='overwrite')

In [100]:
df_zones_final.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [cast(zone#632 as bigint)], [cast(LocationID#869 as bigint)], Inner, BuildRight, false
   :- HashAggregate(keys=[PULocationID#103, _groupingexpression#1007], functions=[sum(total_amount#112), count(1)])
   :  +- Exchange hashpartitioning(PULocationID#103, _groupingexpression#1007, 200), ENSURE_REQUIREMENTS, [plan_id=3405]
   :     +- HashAggregate(keys=[PULocationID#103, _groupingexpression#1007], functions=[partial_sum(total_amount#112), partial_count(1)])
   :        +- Project [PULocationID#103, total_amount#112, date_trunc(hour, tpep_pickup_datetime#97, Some(Etc/UTC)) AS _groupingexpression#1007]
   :           +- Filter ((isnotnull(tpep_pickup_datetime#97) AND (tpep_pickup_datetime#97 >= 2020-01-01 00:00:00)) AND isnotnull(PULocationID#103))
   :              +- FileScan csv [tpep_pickup_datetime#97,PULocationID#103,total_amount#112] Batched: false, DataFilters: [isnotnull(tpep_pickup_datetime#97), (tpep_

# Resilient Distribution Datasets (RDD)

In [105]:
df_green.rdd.take(10)

[Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2020, 12, 1, 0, 29, 37), lpep_dropoff_datetime=datetime.datetime(2020, 12, 1, 0, 32, 51), store_and_fwd_flag='N', RatecodeID=1, PULocationID=75, DOLocationID=75, passenger_count=1, trip_distance=0.59, fare_amount=4.5, extra=0.5, mta_tax=0.5, tip_amount=1.16, tolls_amount=0.0, ehail_fee=None, improvement_surcharge=0.3, total_amount=6.96, payment_type=1, trip_type=1, congestion_surcharge=0.0),
 Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2020, 12, 1, 0, 41, 46), lpep_dropoff_datetime=datetime.datetime(2020, 12, 1, 0, 46, 31), store_and_fwd_flag='N', RatecodeID=1, PULocationID=75, DOLocationID=74, passenger_count=1, trip_distance=1.24, fare_amount=6.0, extra=0.5, mta_tax=0.5, tip_amount=1.46, tolls_amount=0.0, ehail_fee=None, improvement_surcharge=0.3, total_amount=8.76, payment_type=1, trip_type=1, congestion_surcharge=0.0),
 Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2020, 12, 1, 0, 5, 46), lpep_dropoff_datet

In [109]:
rdd = df_green \
      .select('lpep_pickup_datetime', 'PULocationID', 'total_amount') \
      .rdd

In [110]:
rdd.take(5)

[Row(lpep_pickup_datetime=datetime.datetime(2020, 12, 1, 0, 29, 37), PULocationID=75, total_amount=6.96),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 12, 1, 0, 41, 46), PULocationID=75, total_amount=8.76),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 12, 1, 0, 5, 46), PULocationID=244, total_amount=8.76),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 11, 30, 23, 59, 17), PULocationID=75, total_amount=23.05),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 12, 1, 0, 57, 3), PULocationID=74, total_amount=9.55)]

In [111]:
from datetime import datetime

In [134]:
start = datetime(year=2020, month=1, day=1)

def filter_outliers(row):
  return row.lpep_pickup_datetime >= start

In [135]:
rows = rdd.take(10)
row = rows[0]

In [117]:
row.lpep_pickup_datetime.replace(minute=0, second=0, microsecond=0)

datetime.datetime(2020, 12, 1, 0, 0)

In [136]:
def prepare_grouping(row):
  hour = row.lpep_pickup_datetime.replace(minute=0, second=0, microsecond=0)
  zone = row.PULocationID
  key = (hour, zone)

  amount = row.total_amount
  count = 1
  value = (amount, count)

  return (key, value)

In [141]:
def calculate_revenue(left_value, right_value):
  left_amount, left_count = left_value
  right_amount, right_count = right_value

  output_amount = left_amount + right_amount
  output_count = left_count + right_count
  output_value = (output_amount, output_count)

  return (output_amount, output_count)

In [142]:
from collections import namedtuple

In [143]:
RevenueRow = namedtuple('RevenueRow', ['hour', 'zone', 'revenue', 'count'])

In [144]:
def unwrap(row):
  return RevenueRow (
      hour = row[0][0],
      zone = row[0][1],
      revenue = row[1][0],
      count = row[1][1])

In [150]:
from pyspark.sql import types

In [151]:
df_rdd_schema = types.StructType([
     types.StructField('hour', types.TimestampType(), True),
     types.StructField('zone', types.IntegerType(), True),
     types.StructField('revenue', types.DoubleType(), True),
     types.StructField('count', types.IntegerType(), True)
     ])

In [156]:
df_rdd_result = rdd \
  .filter(filter_outliers) \
  .map(prepare_grouping) \
  .reduceByKey(calculate_revenue) \
  .map(unwrap) \
  .toDF(df_rdd_schema)

In [157]:
df_rdd_result.show()

+-------------------+----+------------------+-----+
|               hour|zone|           revenue|count|
+-------------------+----+------------------+-----+
|2020-12-01 00:00:00|  75|             80.35|    6|
|2020-12-01 00:00:00| 244|22.560000000000002|    2|
|2020-11-30 23:00:00|  75|             23.05|    1|
|2020-12-01 00:00:00|  74|              9.55|    1|
|2020-12-01 00:00:00|  69|               7.3|    1|
|2020-12-01 00:00:00| 116|              10.8|    1|
|2020-12-01 00:00:00|  41|             41.96|    3|
|2020-12-01 00:00:00| 130|              38.1|    2|
|2020-12-01 01:00:00|   7|              9.96|    1|
|2020-12-01 01:00:00| 116|              85.6|    2|
|2020-12-01 00:00:00| 119|             66.42|    2|
|2020-12-01 01:00:00|  74|               5.3|    1|
|2020-12-01 01:00:00|  42|             28.26|    1|
|2020-12-01 01:00:00| 166|               4.3|    1|
|2020-12-01 01:00:00|  41|              22.1|    2|
|2020-12-01 01:00:00| 236|              19.3|    1|
|2020-12-01 

In [158]:
# Save to parquet
df_rdd_result.write.parquet('tmp/green-revenue')

In [159]:
spark.read.parquet('tmp/green-revenue')

DataFrame[hour: timestamp, zone: int, revenue: double, count: int]

### RDD mapPartition

In [160]:
# Scenario: A service that predicts the duration of a trip (ML Model)

columns = ['VendorID', 'lpep_pickup_datetime', 'PULocationID', 'DOLocationID', 'trip_distance']

duration_rdd = df_green \
    .select(columns) \
    .rdd


In [173]:
import pandas as pd

In [174]:
rows = duration_rdd.take(10)

In [175]:
pd.DataFrame(rows, columns=columns)

,VendorID,lpep_pickup_datetime,PULocationID,DOLocationID,trip_distance
0,2,2020-12-01 00:29:37,75,75,0.59
1,2,2020-12-01 00:41:46,75,74,1.24
2,2,2020-12-01 00:05:46,244,243,1.19
3,2,2020-11-30 23:59:17,75,68,5.08
4,2,2020-12-01 00:57:03,74,263,1.24
5,2,2020-12-01 00:05:38,244,74,2.81
6,2,2020-12-01 00:21:19,69,116,1.12
7,2,2020-12-01 00:33:46,116,247,2.27
8,2,2020-12-01 00:05:28,75,213,6.67
9,2,2020-12-01 00:36:58,75,74,0.95


In [178]:
def apply_model_batch(rows):
  df = pd.DataFrame(rows, columns=columns)
  cnt = len(df)

  return [cnt]

In [180]:
duration_rdd.mapPartitions(apply_model_batch).collect()

[83130]